# ETL — Modelo Dimensional Fase 2 (HCH)

Organizado en **Extract → Transform → Load**, por cada dimensión primero
y las tablas de hechos al final (así se resuelven todas las llaves
foráneas antes de cargar los hechos).

**Estrategia: coexistir.** Este notebook solo LEE de las tablas
originales (`bmc_precios`, `importaciones`, `trm_diaria`, etc.) y
ESCRIBE en las tablas nuevas (`Dim*` / `Hechos*`) — no las modifica.

**Es seguro re-ejecutar todo el notebook**: cada carga hace `DELETE`
antes de `INSERT`, no duplica filas.

## 0. Configuración

In [1]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "mercado_hch.db"

def q(sql, params=None):
    """Atajo para SELECT -> DataFrame."""
    with sqlite3.connect(DB_PATH) as con:
        return pd.read_sql(sql, con, params=params)

def run(sql, params=None):
    """Atajo para sentencias que no devuelven filas."""
    with sqlite3.connect(DB_PATH) as con:
        con.execute(sql, params or [])
        con.commit()

def load_append(df, tabla, delete_where=None, delete_params=None):
    """LOAD genérico: opcionalmente borra un subconjunto y agrega el DataFrame."""
    with sqlite3.connect(DB_PATH) as con:
        if delete_where:
            con.execute(f"DELETE FROM {tabla} WHERE {delete_where}", delete_params or [])
        df.to_sql(tabla, con, if_exists='append', index=False)
        con.commit()

def tablas_existentes():
    return set(q("SELECT name FROM sqlite_master WHERE type='table'")['name'])

def to_id_fecha(serie):
    """Convierte una serie de fechas (texto 'YYYY-MM-DD' o timestamp
    completo 'YYYY-MM-DD HH:MM:SS') al id_fecha entero 'YYYYMMDD' de
    DimFecha, de forma robusta sin importar el formato exacto de entrada."""
    return pd.to_datetime(serie).dt.strftime('%Y%m%d').astype(int)

# ── Alias de insumos entre fuentes ────────────────────────────────
# BMC y Comtrade a veces nombran el mismo insumo distinto (ej. BMC usa
# 'fosfato_monodicalcico', Comtrade usa el código genérico 'fosfato').
# Este diccionario normaliza todo a UN solo codigo_insumo canónico antes
# de cargar DimInsumo y de resolver la FK en HechosPrecioInsumo — así no
# quedan dos filas de dimensión para el mismo insumo real.
# Formato: {nombre_como_aparece_en_la_fuente: nombre_canonico}
ALIAS_INSUMO = {
    'fosfato_monodicalcico': 'fosfato',
}

# ── Override de categoría para producto terminado ──────────────────
# concentrado_aves/cerdos/perros/gatos NO son insumos sustitutos de HCH
# (materia prima) — son producto terminado que compra el ganadero/dueño
# de mascota. Se excluyen de la categoría 'proteico' por defecto para no
# contaminar el análisis de sustitución (2.a/2.b/2.c del cuadro de negocio).
CATEGORIA_OVERRIDE_INSUMO = {
    'concentrado_aves':   'producto_terminado',
    'concentrado_cerdos': 'producto_terminado',
    'concentrado_perros': 'producto_terminado',
    'concentrado_gatos':  'producto_terminado',
}

def canonicalizar_insumo(serie):
    return serie.replace(ALIAS_INSUMO)

print("✅ Conectado a", DB_PATH)
print("Tablas actuales:", sorted(tablas_existentes()))


✅ Conectado a mercado_hch.db
Tablas actuales: ['DimFecha', 'DimFuente', 'DimIndicador', 'DimInsumo', 'DimPais', 'HechosIndicadorMacro', 'HechosPrecioInsumo', 'bmc_precios', 'bmc_precios_correcciones_log', 'bmc_precios_limpio', 'cme_diario', 'dian_2026', 'hch_forecast_xgb_v2', 'hch_precio_unificado', 'importaciones', 'ingredientes_comparacion', 'roni_mensual', 'sacrificio_bovino', 'sqlite_sequence', 'trm_diaria']


## 1. DDL — crear las 7 tablas del esquema dimensional (idempotente)

In [2]:
DDL = """
CREATE TABLE IF NOT EXISTS DimFecha (
    id_fecha        INTEGER PRIMARY KEY,
    fecha           TEXT NOT NULL UNIQUE,
    anio            INTEGER NOT NULL,
    trimestre       INTEGER NOT NULL,
    mes             INTEGER NOT NULL,
    nombre_mes      TEXT NOT NULL,
    dia             INTEGER NOT NULL,
    dia_semana      TEXT NOT NULL,
    es_fin_semana   INTEGER NOT NULL DEFAULT 0,
    es_festivo      INTEGER NOT NULL DEFAULT 0
);

CREATE TABLE IF NOT EXISTS DimInsumo (
    id_insumo       INTEGER PRIMARY KEY AUTOINCREMENT,
    codigo_insumo   TEXT NOT NULL UNIQUE,
    nombre_insumo   TEXT NOT NULL,
    categoria       TEXT,
    unidad_base     TEXT NOT NULL DEFAULT 'ton'
);

CREATE TABLE IF NOT EXISTS DimFuente (
    id_fuente       INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre_fuente   TEXT NOT NULL UNIQUE,
    tipo            TEXT NOT NULL,
    frecuencia      TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS DimPais (
    id_pais         INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre_pais     TEXT NOT NULL UNIQUE,
    region          TEXT
);

CREATE TABLE IF NOT EXISTS DimIndicador (
    id_indicador    INTEGER PRIMARY KEY AUTOINCREMENT,
    codigo_indicador TEXT NOT NULL UNIQUE,
    nombre_indicador TEXT NOT NULL,
    categoria       TEXT,
    unidad_medida   TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS HechosPrecioInsumo (
    id_hecho         INTEGER PRIMARY KEY AUTOINCREMENT,
    id_fecha         INTEGER NOT NULL REFERENCES DimFecha(id_fecha),
    id_insumo        INTEGER NOT NULL REFERENCES DimInsumo(id_insumo),
    id_fuente        INTEGER NOT NULL REFERENCES DimFuente(id_fuente),
    id_pais          INTEGER NOT NULL REFERENCES DimPais(id_pais),
    codigo_reporte   TEXT,
    cantidad_kg      REAL,
    valor_total_cop  REAL,
    precio_usd_ton   REAL,
    precio_cop_ton   REAL
);

CREATE TABLE IF NOT EXISTS HechosIndicadorMacro (
    id_hecho         INTEGER PRIMARY KEY AUTOINCREMENT,
    id_fecha         INTEGER NOT NULL REFERENCES DimFecha(id_fecha),
    id_indicador     INTEGER NOT NULL REFERENCES DimIndicador(id_indicador),
    valor            REAL
);
"""

# Nota: las tablas de hechos se vuelven a recrear (DROP + CREATE) dentro
# de sus propias celdas de Load (secciones 7.3 y 8.3) cada vez que se
# corren, así que aquí basta con CREATE TABLE IF NOT EXISTS.
with sqlite3.connect(DB_PATH) as con:
    con.executescript(DDL)
    con.commit()

print("✅ Esquema dimensional creado/verificado")
print([t for t in tablas_existentes() if t.startswith(('Dim', 'Hechos'))])


✅ Esquema dimensional creado/verificado
['DimFecha', 'HechosIndicadorMacro', 'DimPais', 'DimIndicador', 'DimFuente', 'HechosPrecioInsumo', 'DimInsumo']


---
## 2. Dimensión `DimFecha`

### 2.1 Extract
No hay fuente externa: se genera el calendario completo 2008–2027.

In [3]:
try:
    import holidays
    festivos_co = holidays.Colombia(years=range(2008, 2028))
except ImportError:
    print("⚠️  Librería 'holidays' no instalada. Corre: pip install holidays")
    print("   Continuando con es_festivo = 0 para todas las fechas.")
    festivos_co = {}

rango_fechas = pd.date_range('2008-01-01', '2027-12-31', freq='D')
print(f"Extract DimFecha: {len(rango_fechas):,} días generados ({rango_fechas.min().date()} → {rango_fechas.max().date()})")


Extract DimFecha: 7,305 días generados (2008-01-01 → 2027-12-31)


### 2.2 Transform
Derivar atributos de calendario (año, trimestre, mes, día de semana, festivo).

In [4]:
DIAS_ES = ['lunes','martes','miércoles','jueves','viernes','sábado','domingo']
MESES_ES = ['enero','febrero','marzo','abril','mayo','junio','julio',
            'agosto','septiembre','octubre','noviembre','diciembre']

df_fecha = pd.DataFrame({'fecha_dt': rango_fechas})
df_fecha['id_fecha']      = df_fecha['fecha_dt'].dt.strftime('%Y%m%d').astype(int)
df_fecha['fecha']         = df_fecha['fecha_dt'].dt.strftime('%Y-%m-%d')
df_fecha['anio']          = df_fecha['fecha_dt'].dt.year
df_fecha['trimestre']     = df_fecha['fecha_dt'].dt.quarter
df_fecha['mes']           = df_fecha['fecha_dt'].dt.month
df_fecha['nombre_mes']    = df_fecha['mes'].apply(lambda m: MESES_ES[m-1])
df_fecha['dia']           = df_fecha['fecha_dt'].dt.day
df_fecha['dia_semana']    = df_fecha['fecha_dt'].dt.weekday.apply(lambda d: DIAS_ES[d])
df_fecha['es_fin_semana'] = (df_fecha['fecha_dt'].dt.weekday >= 5).astype(int)
df_fecha['es_festivo']    = df_fecha['fecha_dt'].dt.date.apply(lambda d: int(d in festivos_co))

df_fecha = df_fecha.drop(columns=['fecha_dt'])
print(f"Transform DimFecha: {len(df_fecha):,} filas listas, {df_fecha['es_festivo'].sum()} festivos marcados")
df_fecha.head(3)


Transform DimFecha: 7,305 filas listas, 356 festivos marcados


,id_fecha,fecha,anio,trimestre,mes,nombre_mes,dia,dia_semana,es_fin_semana,es_festivo
0,20080101,2008-01-01,2008,1,1,enero,1,martes,0,1
1,20080102,2008-01-02,2008,1,1,enero,2,miércoles,0,0
2,20080103,2008-01-03,2008,1,1,enero,3,jueves,0,0


### 2.3 Load

In [5]:
load_append(df_fecha, 'DimFecha', delete_where='1=1')
print(f"✅ DimFecha cargada: {len(df_fecha):,} filas")


✅ DimFecha cargada: 7,305 filas


---
## 3. Dimensión `DimFuente`

### 3.1 Extract
Catálogo estático — no depende de ninguna tabla de origen.

In [6]:
fuentes_raw = [
    ('BMC',      'nacional',    'semanal'),
    ('Comtrade', 'importacion', 'anual'),
]
print(f"Extract DimFuente: {len(fuentes_raw)} fuentes definidas")


Extract DimFuente: 2 fuentes definidas


### 3.2 Transform
No requiere transformación adicional (catálogo ya normalizado).

In [7]:
df_fuente = pd.DataFrame(fuentes_raw, columns=['nombre_fuente', 'tipo', 'frecuencia'])
df_fuente


,nombre_fuente,tipo,frecuencia
0,BMC,nacional,semanal
1,Comtrade,importacion,anual


### 3.3 Load

In [8]:
for _, row in df_fuente.iterrows():
    run("INSERT OR IGNORE INTO DimFuente (nombre_fuente, tipo, frecuencia) VALUES (?, ?, ?)",
        [row['nombre_fuente'], row['tipo'], row['frecuencia']])
print("✅ DimFuente cargada")
print(q("SELECT * FROM DimFuente"))


✅ DimFuente cargada


   id_fuente nombre_fuente         tipo frecuencia
0          1           BMC     nacional    semanal
1          2      Comtrade  importacion      anual


---
## 4. Dimensión `DimIndicador`

### 4.1 Extract
Catálogo estático de los 10 indicadores monitoreados (macro + demanda).

In [9]:
indicadores_raw = [
    ('trm',               'Tasa Representativa del Mercado', 'cambiario',  'COP/USD'),
    ('cme_soya',          'Commodity harina de soya (CME)',  'commodity',  'USD/ton'),
    ('cme_maiz',          'Commodity grano de maíz (CME)',   'commodity',  'USD/ton'),
    ('roni',              'Índice El Niño/La Niña (RONI)',   'climatico',  'indice'),
    ('sacrificio_bovino', 'Sacrificio bovino potencial',     'oferta',     'cabezas'),
    ('demanda_porcina',   'Sacrificio porcino (demanda potencial de concentrado)', 'demanda', 'cabezas'),
    ('demanda_concentrado_avicola', 'Consumo de concentrado avícola (BMC)', 'demanda', 'ton/mes'),
    ('demanda_concentrado_porcino', 'Consumo de concentrado porcino (BMC)', 'demanda', 'ton/mes'),
    ('demanda_concentrado_perros',  'Consumo de concentrado para perros (BMC)', 'demanda', 'ton/mes'),
    ('demanda_concentrado_gatos',   'Consumo de concentrado para gatos (BMC)', 'demanda', 'ton/mes'),
]
print(f"Extract DimIndicador: {len(indicadores_raw)} indicadores definidos")


Extract DimIndicador: 10 indicadores definidos


### 4.2 Transform
No requiere transformación adicional.

In [10]:
df_indicador = pd.DataFrame(indicadores_raw,
    columns=['codigo_indicador', 'nombre_indicador', 'categoria', 'unidad_medida'])
df_indicador


,codigo_indicador,nombre_indicador,categoria,unidad_medida
0,trm,Tasa Representativa del Mercado,cambiario,COP/USD
1,cme_soya,Commodity harina de soya (CME),commodity,USD/ton
2,cme_maiz,Commodity grano de maíz (CME),commodity,USD/ton
3,roni,Índice El Niño/La Niña (RONI),climatico,indice
4,sacrificio_bovino,Sacrificio bovino potencial,oferta,cabezas
5,demanda_porcina,Sacrificio porcino (demanda potencial de conce...,demanda,cabezas
6,demanda_concentrado_avicola,Consumo de concentrado avícola (BMC),demanda,ton/mes
7,demanda_concentrado_porcino,Consumo de concentrado porcino (BMC),demanda,ton/mes
8,demanda_concentrado_perros,Consumo de concentrado para perros (BMC),demanda,ton/mes
9,demanda_concentrado_gatos,Consumo de concentrado para gatos (BMC),demanda,ton/mes


### 4.3 Load

In [11]:
for _, row in df_indicador.iterrows():
    run("""INSERT OR IGNORE INTO DimIndicador
             (codigo_indicador, nombre_indicador, categoria, unidad_medida) VALUES (?, ?, ?, ?)""",
        [row['codigo_indicador'], row['nombre_indicador'], row['categoria'], row['unidad_medida']])
print("✅ DimIndicador cargada")
print(q("SELECT * FROM DimIndicador"))


✅ DimIndicador cargada
   id_indicador             codigo_indicador  \
0             1                          trm   
1             2                     cme_soya   
2             3                     cme_maiz   
3             4                         roni   
4             5            sacrificio_bovino   
5            26              demanda_porcina   
6            33  demanda_concentrado_avicola   
7            34  demanda_concentrado_porcino   
8            35   demanda_concentrado_perros   
9            36    demanda_concentrado_gatos   

                                    nombre_indicador  categoria unidad_medida  
0                    Tasa Representativa del Mercado  cambiario       COP/USD  
1                     Commodity harina de soya (CME)  commodity       USD/ton  
2                      Commodity grano de maíz (CME)  commodity       USD/ton  
3                      Índice El Niño/La Niña (RONI)  climatico        indice  
4                        Sacrificio bovino poten

---
## 5. Dimensión `DimPais`

### 5.1 Extract
Países reales desde `importaciones` (Comtrade con desglose por `pais_origen`).

In [12]:
existentes = tablas_existentes()

if 'importaciones' in existentes:
    df_pais_raw = q("""SELECT DISTINCT pais_origen FROM importaciones
                         WHERE pais_origen IS NOT NULL""")
    print(f"Extract DimPais: {len(df_pais_raw)} países distintos encontrados en 'importaciones'")
else:
    df_pais_raw = pd.DataFrame(columns=['pais_origen'])
    print("⚠️  Tabla 'importaciones' no encontrada — DimPais solo tendrá la fila comodín")


Extract DimPais: 36 países distintos encontrados en 'importaciones'


### 5.2 Transform
- Excluir `'Mundo (agregado)'` (decisión tomada: no se usa en los hechos, evita doble conteo)
- Agregar la fila comodín `id_pais = -1` para insumos de mercado nacional (BMC) que no tienen país de origen

In [13]:
df_pais = df_pais_raw[df_pais_raw['pais_origen'] != 'Mundo (agregado)'].copy()
df_pais = df_pais.rename(columns={'pais_origen': 'nombre_pais'})
df_pais['region'] = None

# Fila comodín (se maneja aparte porque tiene id fijo = -1)
print(f"Transform DimPais: {len(df_pais)} países reales + 1 fila comodín")
df_pais.head()


Transform DimPais: 35 países reales + 1 fila comodín


,nombre_pais,region
0,Argentina,None
1,Brasil,None
2,Italia,None
3,Paraguay,None
4,México,None


### 5.3 Load

In [14]:
run("INSERT OR IGNORE INTO DimPais (id_pais, nombre_pais, region) VALUES (-1, 'No aplica', 'No aplica')")

for _, row in df_pais.iterrows():
    run("INSERT OR IGNORE INTO DimPais (nombre_pais, region) VALUES (?, ?)",
        [row['nombre_pais'], row['region']])

print(f"✅ DimPais cargada: {len(q('SELECT * FROM DimPais'))} filas totales (incluye comodín)")
print(q("SELECT * FROM DimPais ORDER BY id_pais LIMIT 10"))


✅ DimPais cargada: 36 filas totales (incluye comodín)
   id_pais                                        nombre_pais     region
0       -1                                          No aplica  No aplica
1        1                                          Argentina       None
2        2                                             Brasil       None
3        3                                             Italia       None
4        4                                           Paraguay       None
5        5                                             México       None
6        6  Estados Unidos (código Comtrade — incluye Puer...       None
7        7                                             España       None
8        8                                            Uruguay       None
9        9                                          Guatemala       None


---
## 6. Dimensión `DimInsumo`

### 6.1 Extract
Productos reales desde BMC (`bmc_precios_limpio`/`bmc_precios`) y Comtrade (`importaciones`).

In [15]:
bmc_tabla = 'bmc_precios_limpio' if 'bmc_precios_limpio' in existentes else 'bmc_precios'

productos_bmc = q(f"SELECT DISTINCT producto FROM {bmc_tabla}")['producto'].tolist() \
    if bmc_tabla in existentes else []
productos_ct = q("SELECT DISTINCT producto FROM importaciones")['producto'].tolist() \
    if 'importaciones' in existentes else []

# Normalizar alias ANTES de deduplicar (ej. 'fosfato_monodicalcico' -> 'fosfato')
productos_bmc = [ALIAS_INSUMO.get(p, p) for p in productos_bmc]
productos_ct  = [ALIAS_INSUMO.get(p, p) for p in productos_ct]

print(f"Extract DimInsumo: {len(productos_bmc)} productos desde {bmc_tabla!r}, "
      f"{len(productos_ct)} productos desde 'importaciones' (ya normalizados por alias)")


Extract DimInsumo: 14 productos desde 'bmc_precios_limpio', 5 productos desde 'importaciones' (ya normalizados por alias)


### 6.2 Transform
- Unir y deduplicar productos de ambas fuentes
- Clasificar categoría por palabra clave (familia soya → proteico por defecto; sebo/grasa/maíz/aceite → energético; fosfato → mineral)
- Aplicar override para producto terminado (concentrado_aves/cerdos/perros/gatos)

In [16]:
def categoria_de(nombre):
    n = nombre.lower()
    # Toda la familia de soya (grano/harina/torta) queda 'proteico' por
    # defecto: aunque el grano entero aporta algo de energía por el aceite,
    # su rol dominante en formulación sigue siendo proteico.
    if any(k in n for k in ['sebo', 'grasa', 'maiz', 'aceite']):
        return 'energetico'
    if 'fosfato' in n:
        return 'mineral'
    return 'proteico'

productos = sorted(set(productos_bmc) | set(productos_ct))

df_insumo = pd.DataFrame({'codigo_insumo': productos})
df_insumo['nombre_insumo'] = df_insumo['codigo_insumo'].str.replace('_', ' ').str.title()
df_insumo['categoria']     = df_insumo['codigo_insumo'].apply(categoria_de)
df_insumo['categoria']     = df_insumo.apply(
    lambda r: CATEGORIA_OVERRIDE_INSUMO.get(r['codigo_insumo'], r['categoria']), axis=1)
df_insumo['unidad_base']   = 'ton'

print(f"Transform DimInsumo: {len(df_insumo)} insumos únicos")
df_insumo


Transform DimInsumo: 17 insumos únicos


,codigo_insumo,nombre_insumo,categoria,unidad_base
0,HCH,Hch,proteico,ton
1,concentrado_aves,Concentrado Aves,producto_terminado,ton
2,concentrado_cerdos,Concentrado Cerdos,producto_terminado,ton
3,concentrado_gatos,Concentrado Gatos,producto_terminado,ton
4,concentrado_perros,Concentrado Perros,producto_terminado,ton
5,fosfato,Fosfato,mineral,ton
6,grano_soya,Grano Soya,proteico,ton
7,harina_carne,Harina Carne,proteico,ton
8,harina_hueso,Harina Hueso,proteico,ton
9,harina_pescado,Harina Pescado,proteico,ton


### 6.3 Load

In [17]:
for _, row in df_insumo.iterrows():
    run("""INSERT OR IGNORE INTO DimInsumo (codigo_insumo, nombre_insumo, categoria, unidad_base)
             VALUES (?, ?, ?, ?)""",
        [row['codigo_insumo'], row['nombre_insumo'], row['categoria'], row['unidad_base']])

print(f"✅ DimInsumo cargada: {len(q('SELECT * FROM DimInsumo'))} filas")
print(q("SELECT * FROM DimInsumo ORDER BY id_insumo"))


✅ DimInsumo cargada: 18 filas
    id_insumo          codigo_insumo          nombre_insumo   categoria  \
0           1                    HCH                    Hch    proteico   
1           2       concentrado_aves       Concentrado Aves    proteico   
2           3     concentrado_cerdos     Concentrado Cerdos    proteico   
3           4      concentrado_gatos      Concentrado Gatos    proteico   
4           5     concentrado_perros     Concentrado Perros    proteico   
5           6                fosfato                Fosfato     mineral   
6           7  fosfato_monodicalcico  Fosfato Monodicalcico     mineral   
7           8             grano_soya             Grano Soya  energetico   
8           9           harina_carne           Harina Carne    proteico   
9          10           harina_hueso           Harina Hueso    proteico   
10         11         harina_pescado         Harina Pescado    proteico   
11         12    harina_pluma_sangre    Harina Pluma Sangre    proteic

---
## 7. Tabla de hechos `HechosPrecioInsumo`

Depende de `DimFecha`, `DimInsumo`, `DimFuente`, `DimPais` — ya cargadas arriba.

### 7.1 Extract
Lee BMC (diario, nacional) y Comtrade (anual, con país) por separado.

In [18]:
df_bmc_raw = q(f"""
    SELECT fecha, producto, n_rueda, cantidad_kg, total_cop, precio_usd_ton, precio_cop_kg
    FROM {bmc_tabla}
    WHERE fecha IS NOT NULL AND producto IS NOT NULL
""") if bmc_tabla in existentes else pd.DataFrame()

df_ct_raw = q("""
    SELECT refYear, producto, pais_origen, cmdCode, netWgt, cifvalue, precio_usd_ton
    FROM importaciones
    WHERE pais_origen != 'Mundo (agregado)'
      AND refYear IS NOT NULL AND producto IS NOT NULL
""") if 'importaciones' in existentes else pd.DataFrame()

print(f"Extract HechosPrecioInsumo: {len(df_bmc_raw)} filas BMC, {len(df_ct_raw)} filas Comtrade")


Extract HechosPrecioInsumo: 8788 filas BMC, 516 filas Comtrade


### 7.2 Transform
- Resolver llaves foráneas contra las dimensiones ya cargadas
- BMC: grano diario real, `id_pais = -1` (mercado nacional)
- Comtrade: grano anual → se mapea al 1° de enero de `refYear` por convención;
  `valor_total_cop` se aproxima con el TRM promedio del año (Comtrade reporta en USD)

In [19]:
id_insumo_map = dict(q("SELECT codigo_insumo, id_insumo FROM DimInsumo").values)
id_pais_map   = dict(q("SELECT nombre_pais, id_pais FROM DimPais").values)
id_fuente_bmc = q("SELECT id_fuente FROM DimFuente WHERE nombre_fuente='BMC'").iloc[0, 0]
id_fuente_ct  = q("SELECT id_fuente FROM DimFuente WHERE nombre_fuente='Comtrade'").iloc[0, 0]
fechas_validas = set(q("SELECT id_fecha FROM DimFecha")['id_fecha'])

# ── BMC ────────────────────────────────────────────────────────
df_bmc = df_bmc_raw.copy()
if not df_bmc.empty:
    df_bmc['id_fecha']        = to_id_fecha(df_bmc['fecha'])
    df_bmc['id_insumo']       = canonicalizar_insumo(df_bmc['producto']).map(id_insumo_map)
    df_bmc['id_fuente']       = id_fuente_bmc
    df_bmc['id_pais']         = -1
    df_bmc['codigo_reporte']  = df_bmc['n_rueda'].astype(str)
    df_bmc['valor_total_cop'] = df_bmc['total_cop']
    df_bmc['precio_cop_ton']  = df_bmc['precio_cop_kg'] * 1000

    df_bmc = df_bmc[['id_fecha','id_insumo','id_fuente','id_pais','codigo_reporte',
                      'cantidad_kg','valor_total_cop','precio_usd_ton','precio_cop_ton']]
    antes = len(df_bmc)
    df_bmc = df_bmc.dropna(subset=['id_insumo'])
    df_bmc = df_bmc[df_bmc['id_fecha'].isin(fechas_validas)]
    if antes != len(df_bmc):
        print(f"⚠️  {antes - len(df_bmc)} filas BMC descartadas (insumo sin mapear o fecha fuera de rango)")

# ── Comtrade ───────────────────────────────────────────────────
df_ct = df_ct_raw.copy()
if not df_ct.empty:
    trm_anual = q("""SELECT SUBSTR(fecha,1,4) AS anio, AVG(trm_cop_usd) AS trm_prom
                       FROM trm_diaria GROUP BY anio""") if 'trm_diaria' in existentes \
                else pd.DataFrame(columns=['anio','trm_prom'])
    trm_map = dict(zip(trm_anual['anio'].astype(int), trm_anual['trm_prom']))

    df_ct['id_fecha']        = (df_ct['refYear'].astype(int).astype(str) + '0101').astype(int)
    df_ct['id_insumo']       = canonicalizar_insumo(df_ct['producto']).map(id_insumo_map)
    df_ct['id_fuente']       = id_fuente_ct
    df_ct['id_pais']         = df_ct['pais_origen'].map(id_pais_map)
    df_ct['codigo_reporte']  = df_ct['cmdCode'].astype(str)
    df_ct['cantidad_kg']     = df_ct['netWgt']
    trm_aprox                = df_ct['refYear'].astype(int).map(trm_map)
    df_ct['valor_total_cop'] = df_ct['cifvalue'] * trm_aprox
    df_ct['precio_cop_ton']  = df_ct['precio_usd_ton'] * trm_aprox

    df_ct = df_ct[['id_fecha','id_insumo','id_fuente','id_pais','codigo_reporte',
                    'cantidad_kg','valor_total_cop','precio_usd_ton','precio_cop_ton']]
    df_ct = df_ct.dropna(subset=['id_insumo', 'id_pais'])
    if not trm_map:
        print("ℹ️  'trm_diaria' no encontrada — valor_total_cop / precio_cop_ton quedaron NULL para Comtrade")

print(f"Transform HechosPrecioInsumo: {len(df_bmc)} filas BMC listas, {len(df_ct)} filas Comtrade listas")

# ── Diagnóstico informativo: filas que comparten la misma llave de negocio
# (fecha+insumo+fuente+país+codigo_reporte). No se bloquea la carga —son
# lotes/vendedores distintos dentro del mismo evento— pero es útil saberlo.
for nombre, df in [('BMC', df_bmc), ('Comtrade', df_ct)]:
    if df.empty:
        continue
    llave = ['id_fecha', 'id_insumo', 'id_fuente', 'id_pais', 'codigo_reporte']
    n_dup = df.duplicated(subset=llave, keep=False).sum()
    if n_dup:
        print(f"ℹ️  {nombre}: {n_dup} filas comparten llave de negocio con al menos otra "
              f"(múltiples lotes/vendedores el mismo día) — se cargan todas, no se descartan")


Transform HechosPrecioInsumo: 8788 filas BMC listas, 516 filas Comtrade listas
ℹ️  Comtrade: 435 filas comparten llave de negocio con al menos otra (múltiples lotes/vendedores el mismo día) — se cargan todas, no se descartan


### 7.3 Load

In [20]:
# Recrear siempre la tabla al inicio de la carga (evita cualquier UNIQUE
# viejo, incluyendo índices sueltos que no aparecen en el texto del CREATE
# TABLE). Es seguro: ambas fuentes se vuelven a cargar justo después.
run("DROP TABLE IF EXISTS HechosPrecioInsumo")
run("""CREATE TABLE HechosPrecioInsumo (
    id_hecho         INTEGER PRIMARY KEY AUTOINCREMENT,
    id_fecha         INTEGER NOT NULL REFERENCES DimFecha(id_fecha),
    id_insumo        INTEGER NOT NULL REFERENCES DimInsumo(id_insumo),
    id_fuente        INTEGER NOT NULL REFERENCES DimFuente(id_fuente),
    id_pais          INTEGER NOT NULL REFERENCES DimPais(id_pais),
    codigo_reporte   TEXT,
    cantidad_kg      REAL,
    valor_total_cop  REAL,
    precio_usd_ton   REAL,
    precio_cop_ton   REAL
)""")

if not df_bmc.empty:
    load_append(df_bmc, 'HechosPrecioInsumo', delete_where='id_fuente = ?', delete_params=[int(id_fuente_bmc)])
    print(f"✅ HechosPrecioInsumo: {len(df_bmc):,} filas cargadas desde BMC")

if not df_ct.empty:
    load_append(df_ct, 'HechosPrecioInsumo', delete_where='id_fuente = ?', delete_params=[int(id_fuente_ct)])
    print(f"✅ HechosPrecioInsumo: {len(df_ct):,} filas cargadas desde Comtrade")


✅ HechosPrecioInsumo: 8,788 filas cargadas desde BMC


✅ HechosPrecioInsumo: 516 filas cargadas desde Comtrade


---
## 8. Tabla de hechos `HechosIndicadorMacro`

Depende de `DimFecha` y `DimIndicador` — ya cargadas arriba.

### 8.1 Extract
Lee las 4 tablas fuente de indicadores macro (TRM, CME, RONI, sacrificio bovino).

In [21]:
fuentes_indicador = {
    'trm':               ('trm_diaria',        'fecha',    'trm_cop_usd'),
    'cme_soya':          ('cme_diario',         'fecha',    'harina_soya_cme_usd_ton'),
    'cme_maiz':          ('cme_diario',         'fecha',    'grano_maiz_cme_usd_ton'),
    'roni':              ('roni_mensual',       'anio_mes', 'roni'),
    'sacrificio_bovino': ('sacrificio_bovino',  'fecha',    'potencial_hch'),
    'demanda_porcina':   ('sacrificio_bovino',  'fecha',    'porcinos_cabezas'),
}

extracts = {}
for codigo, (tabla, col_fecha, col_valor) in fuentes_indicador.items():
    if tabla in existentes:
        extracts[codigo] = q(f"SELECT {col_fecha} AS fecha_raw, {col_valor} AS valor "
                              f"FROM {tabla} WHERE {col_valor} IS NOT NULL")
        print(f"Extract '{codigo}': {len(extracts[codigo])} filas desde {tabla}")
    else:
        print(f"⚠️  Tabla '{tabla}' no encontrada, se omite '{codigo}'")


Extract 'trm': 1671 filas desde trm_diaria
Extract 'cme_soya': 1149 filas desde cme_diario
Extract 'cme_maiz': 1149 filas desde cme_diario
Extract 'roni': 917 filas desde roni_mensual
Extract 'sacrificio_bovino': 210 filas desde sacrificio_bovino
Extract 'demanda_porcina': 210 filas desde sacrificio_bovino


### 8.1b Extract adicional — demanda de concentrados terminados (BMC)
Estos 4 indicadores no viven en una tabla propia: se derivan agregando
`cantidad_kg` de `bmc_precios_limpio` por producto, de semanal a mensual
(`resample('MS').sum()`). Grano de negocio: F9 (ver notebook de
Entendimiento y Calidad de Datos).

In [22]:
productos_demanda = {
    'demanda_concentrado_avicola': 'concentrado_aves',
    'demanda_concentrado_porcino': 'concentrado_cerdos',
    'demanda_concentrado_perros':  'concentrado_perros',
    'demanda_concentrado_gatos':   'concentrado_gatos',
}

if bmc_tabla in existentes:
    for codigo, producto in productos_demanda.items():
        df_prod = q(f"SELECT fecha, cantidad_kg FROM {bmc_tabla} WHERE producto = ?", [producto])
        if df_prod.empty:
            print(f"⚠️  Producto '{producto}' no encontrado en {bmc_tabla}, se omite '{codigo}'")
            continue
        df_prod['fecha'] = pd.to_datetime(df_prod['fecha'])
        mensual = df_prod.set_index('fecha')['cantidad_kg'].resample('MS').sum().reset_index()
        mensual.columns = ['fecha_raw', 'valor']
        mensual['fecha_raw'] = mensual['fecha_raw'].dt.strftime('%Y-%m-%d')
        extracts[codigo] = mensual
        print(f"Extract '{codigo}': {len(mensual)} meses agregados desde {bmc_tabla} ({producto})")
else:
    print(f"⚠️  Tabla '{bmc_tabla}' no encontrada, se omiten los 4 indicadores de demanda de concentrados")


Extract 'demanda_concentrado_avicola': 45 meses agregados desde bmc_precios_limpio (concentrado_aves)
Extract 'demanda_concentrado_porcino': 45 meses agregados desde bmc_precios_limpio (concentrado_cerdos)
Extract 'demanda_concentrado_perros': 45 meses agregados desde bmc_precios_limpio (concentrado_perros)
Extract 'demanda_concentrado_gatos': 45 meses agregados desde bmc_precios_limpio (concentrado_gatos)


### 8.2 Transform
- `roni_mensual` usa `anio_mes` ('YYYY-MM') → se completa a `'YYYY-MM-01'`
- Se resuelve `id_indicador` y `id_fecha`, y se descartan fechas fuera del rango de `DimFecha`

In [23]:
id_indicador_map = dict(q("SELECT codigo_indicador, id_indicador FROM DimIndicador").values)

transforms = {}
for codigo, df in extracts.items():
    df = df.copy()
    if codigo == 'roni':
        df['fecha_raw'] = df['fecha_raw'] + '-01'

    df['id_fecha']     = to_id_fecha(df['fecha_raw'])
    df['id_indicador'] = id_indicador_map[codigo]

    antes = len(df)
    df = df[df['id_fecha'].isin(fechas_validas)]
    if antes != len(df):
        print(f"⚠️  '{codigo}': {antes - len(df)} filas descartadas por fecha fuera de rango")

    transforms[codigo] = df[['id_fecha', 'id_indicador', 'valor']]
    print(f"Transform '{codigo}': {len(transforms[codigo])} filas listas")


Transform 'trm': 1671 filas listas


Transform 'cme_soya': 1149 filas listas
Transform 'cme_maiz': 1149 filas listas
⚠️  'roni': 696 filas descartadas por fecha fuera de rango
Transform 'roni': 221 filas listas
Transform 'sacrificio_bovino': 210 filas listas
Transform 'demanda_porcina': 210 filas listas
Transform 'demanda_concentrado_avicola': 45 filas listas
Transform 'demanda_concentrado_porcino': 45 filas listas
Transform 'demanda_concentrado_perros': 45 filas listas
Transform 'demanda_concentrado_gatos': 45 filas listas


### 8.3 Load

In [24]:
run("DROP TABLE IF EXISTS HechosIndicadorMacro")
run("""CREATE TABLE HechosIndicadorMacro (
    id_hecho         INTEGER PRIMARY KEY AUTOINCREMENT,
    id_fecha         INTEGER NOT NULL REFERENCES DimFecha(id_fecha),
    id_indicador     INTEGER NOT NULL REFERENCES DimIndicador(id_indicador),
    valor            REAL
)""")

for codigo, df in transforms.items():
    id_ind = int(id_indicador_map[codigo])
    load_append(df, 'HechosIndicadorMacro', delete_where='id_indicador = ?', delete_params=[id_ind])
    print(f"✅ HechosIndicadorMacro: {len(df):,} filas cargadas para '{codigo}'")


✅ HechosIndicadorMacro: 1,671 filas cargadas para 'trm'
✅ HechosIndicadorMacro: 1,149 filas cargadas para 'cme_soya'
✅ HechosIndicadorMacro: 1,149 filas cargadas para 'cme_maiz'
✅ HechosIndicadorMacro: 221 filas cargadas para 'roni'
✅ HechosIndicadorMacro: 210 filas cargadas para 'sacrificio_bovino'
✅ HechosIndicadorMacro: 210 filas cargadas para 'demanda_porcina'
✅ HechosIndicadorMacro: 45 filas cargadas para 'demanda_concentrado_avicola'
✅ HechosIndicadorMacro: 45 filas cargadas para 'demanda_concentrado_porcino'
✅ HechosIndicadorMacro: 45 filas cargadas para 'demanda_concentrado_perros'


✅ HechosIndicadorMacro: 45 filas cargadas para 'demanda_concentrado_gatos'


---
## 9. Validación cruzada — comparar tablas viejas vs. nuevas

In [25]:
print("=== HechosPrecioInsumo vs. tablas originales ===")
resumen_nuevo = q("""
    SELECT f.nombre_fuente, COUNT(*) AS filas, ROUND(SUM(h.cantidad_kg)/1000, 1) AS ton_total
    FROM HechosPrecioInsumo h JOIN DimFuente f ON h.id_fuente = f.id_fuente
    GROUP BY f.nombre_fuente
""")
print(resumen_nuevo)

if bmc_tabla in existentes:
    orig_bmc = q(f"SELECT COUNT(*) AS filas, ROUND(SUM(cantidad_kg)/1000,1) AS ton_total FROM {bmc_tabla}")
    print(f"\nOriginal {bmc_tabla}: {orig_bmc.iloc[0].to_dict()}")

if 'importaciones' in existentes:
    orig_ct = q("""SELECT COUNT(*) AS filas, ROUND(SUM(netWgt)/1000,1) AS ton_total
                    FROM importaciones WHERE pais_origen != 'Mundo (agregado)'""")
    print(f"Original importaciones (sin 'Mundo'): {orig_ct.iloc[0].to_dict()}")

print("\n=== HechosIndicadorMacro — conteo por indicador ===")
print(q("""
    SELECT i.codigo_indicador, COUNT(*) AS filas, MIN(f.fecha) AS desde, MAX(f.fecha) AS hasta
    FROM HechosIndicadorMacro h
    JOIN DimIndicador i ON h.id_indicador = i.id_indicador
    JOIN DimFecha f ON h.id_fecha = f.id_fecha
    GROUP BY i.codigo_indicador
"""))


=== HechosPrecioInsumo vs. tablas originales ===
  nombre_fuente  filas   ton_total
0           BMC   8788  10541356.7
1      Comtrade    516   7054527.7

Original bmc_precios_limpio: {'filas': 8788.0, 'ton_total': 10541356.7}
Original importaciones (sin 'Mundo'): {'filas': 516.0, 'ton_total': 7054527.7}

=== HechosIndicadorMacro — conteo por indicador ===
              codigo_indicador  filas       desde       hasta
0                     cme_maiz   1149  2022-01-03  2026-07-31
1                     cme_soya   1149  2022-01-03  2026-07-31
2  demanda_concentrado_avicola     45  2022-09-01  2026-05-01
3    demanda_concentrado_gatos     45  2022-09-01  2026-05-01
4   demanda_concentrado_perros     45  2022-09-01  2026-05-01
5  demanda_concentrado_porcino     45  2022-09-01  2026-05-01
6              demanda_porcina    210  2008-10-01  2026-03-01
7                         roni    221  2008-01-01  2026-05-01
8            sacrificio_bovino    210  2008-10-01  2026-03-01
9                    

---
## 10. Vista `variables_modelo_mensual` (pivote ancho, solo HCH)

Pivotea `HechosPrecioInsumo` (precio mensual promedio de HCH, solo BMC)
y `HechosIndicadorMacro` (10 indicadores) a **una fila por mes**, lista para
alimentar el modelo de predicción (análisis 3.a/3.b).

Es una `VIEW` de SQLite, no una tabla física — se recalcula sola cada vez que
la consultas, así que nunca queda desactualizada respecto a los hechos.

**Nota:** los indicadores tienen historia desde 2008, pero TRM/BMC(HCH)/CME
solo arrancan en 2022 — vas a ver `NULL` en esas columnas antes de esa fecha.
Al preparar `X`/`y` para XGBoost, filtra por el rango donde todo esté completo
(`WHERE precio_hch_usd_ton IS NOT NULL`, ver celda de verificación abajo).

In [26]:
VIEW_SQL = """
DROP VIEW IF EXISTS variables_modelo_mensual;
CREATE VIEW variables_modelo_mensual AS
WITH precio_hch_mensual AS (
    SELECT f.anio, f.mes,
           AVG(h.precio_usd_ton) AS precio_hch_usd_ton,
           AVG(h.precio_cop_ton) AS precio_hch_cop_ton
    FROM HechosPrecioInsumo h
    JOIN DimFecha   f  ON h.id_fecha  = f.id_fecha
    JOIN DimInsumo  i  ON h.id_insumo = i.id_insumo
    JOIN DimFuente  fu ON h.id_fuente = fu.id_fuente
    WHERE i.codigo_insumo = 'HCH' AND fu.nombre_fuente = 'BMC'
    GROUP BY f.anio, f.mes
),
indicadores_mensual AS (
    SELECT f.anio, f.mes,
        AVG(CASE WHEN di.codigo_indicador='trm'                          THEN h.valor END) AS trm,
        AVG(CASE WHEN di.codigo_indicador='cme_soya'                     THEN h.valor END) AS cme_soya,
        AVG(CASE WHEN di.codigo_indicador='cme_maiz'                     THEN h.valor END) AS cme_maiz,
        AVG(CASE WHEN di.codigo_indicador='roni'                         THEN h.valor END) AS roni,
        AVG(CASE WHEN di.codigo_indicador='sacrificio_bovino'            THEN h.valor END) AS sacrificio_bovino,
        AVG(CASE WHEN di.codigo_indicador='demanda_porcina'              THEN h.valor END) AS demanda_porcina,
        AVG(CASE WHEN di.codigo_indicador='demanda_concentrado_avicola'  THEN h.valor END) AS demanda_concentrado_avicola,
        AVG(CASE WHEN di.codigo_indicador='demanda_concentrado_porcino'  THEN h.valor END) AS demanda_concentrado_porcino,
        AVG(CASE WHEN di.codigo_indicador='demanda_concentrado_perros'   THEN h.valor END) AS demanda_concentrado_perros,
        AVG(CASE WHEN di.codigo_indicador='demanda_concentrado_gatos'    THEN h.valor END) AS demanda_concentrado_gatos
    FROM HechosIndicadorMacro h
    JOIN DimFecha     f  ON h.id_fecha     = f.id_fecha
    JOIN DimIndicador di ON h.id_indicador = di.id_indicador
    GROUP BY f.anio, f.mes
)
SELECT
    im.anio || '-' || substr('00' || im.mes, -2, 2) AS anio_mes,
    im.anio, im.mes,
    p.precio_hch_usd_ton, p.precio_hch_cop_ton,
    im.trm, im.cme_soya, im.cme_maiz, im.roni,
    im.sacrificio_bovino, im.demanda_porcina,
    im.demanda_concentrado_avicola, im.demanda_concentrado_porcino,
    im.demanda_concentrado_perros, im.demanda_concentrado_gatos
FROM indicadores_mensual im
LEFT JOIN precio_hch_mensual p ON im.anio = p.anio AND im.mes = p.mes
ORDER BY im.anio, im.mes;
"""

with sqlite3.connect(DB_PATH) as con:
    con.executescript(VIEW_SQL)
    con.commit()

print("✅ Vista 'variables_modelo_mensual' creada")


✅ Vista 'variables_modelo_mensual' creada


### Verificación

In [27]:
df_check = q("SELECT * FROM variables_modelo_mensual")
print(f"Total de meses en la vista: {len(df_check)}")

completos = df_check.dropna(subset=['precio_hch_usd_ton'])
print(f"Meses con precio_hch disponible (ventana útil para modelar): {len(completos)}")
print(f"Rango útil: {completos['anio_mes'].min()} → {completos['anio_mes'].max()}")

print("\nÚltimos 6 meses:")
print(completos.tail(6).to_string(index=False))

print("\n% de nulos por columna (dentro de la ventana útil):")
print((completos.isna().mean() * 100).round(1))


Total de meses en la vista: 224
Meses con precio_hch disponible (ventana útil para modelar): 45
Rango útil: 2022-09 → 2026-05

Últimos 6 meses:
anio_mes  anio  mes  precio_hch_usd_ton  precio_hch_cop_ton         trm   cme_soya   cme_maiz  roni  sacrificio_bovino  demanda_porcina  demanda_concentrado_avicola  demanda_concentrado_porcino  demanda_concentrado_perros  demanda_concentrado_gatos
 2025-12  2025   12          435.928333        1.830898e+06 3791.471935 332.641818 173.205455 -0.97           312747.0         702414.0                  40812048.75                  22098734.33                  4391526.90                 1370657.48
 2026-01  2026    1          449.057059        1.886042e+06 3702.999032 324.337000 169.615000 -0.88           288543.0         566027.0                 130655484.36                 127560791.83                 23261621.20                 7876696.55
 2026-02  2026    2          431.126000        1.810733e+06 3675.641429 336.680000 169.110526 -0.71          